# Notebook F2 -- E2: cross-clock image-token residual patching (InternVL3-8B descriptive)
Pre-registered in `PREREG_MECHANISM.md` Section 6; implementation choices in the deviations log (2026-11-07, third and fourth entries). Headline decoding: greedy, `--rep_penalty 1.0` for Qwen.
`mech_e2.py --parts verify,run`: text fixed to clock A, image A shown, the hidden state of the image-token positions replaced at 8 depths (embedding output + 7 layers) by clock B's (real_B), by a same-minute clock C (control c1) or by
norm-matched noise (c2, sets all / minute_hand), for the token sets all / minute_hand / hour_hand / other; 100 pairs (seed 0). Variants: `standard` (the conversion-test prompt) and `text_light` (hour in the text, minute only in the image).
Gate first (identity patch = 0 changed generations, embedding-level all-token patch == showing image B, patch moves the teacher-forced log-prob, region-mask sanity + **overlay PNGs of 20 images in `results/mech_e2/<model>/overlays/` - look at them**).
Resumable per pair; the zip is refreshed after every step. The first pair prints a time extrapolation (a GUESS).

Roles: InternVL3-2B standard = confirmatory (its text_light variant = positive control); Qwen-7B and InternVL3-8B = descriptive (V3 informational; deviations log 2026-11-08, second entry). Qwen-3B is dropped from E2 (fails the 0.15 floor).

## Inputs
| Dataset | Contents at the dataset ROOT |
|---|---|
| **clock-code** (UPDATED, with `mech_e2.py` and the new `mech_lib.py`) | `adapters.py`, `clocks.py`, `eval_behavior.py`, `describe_check.py`, `text_only_v2.py`, `conversion_test.py`, `cap_check.py`, `mech_lib.py`, `mech_e3.py`, `mech_e1.py`, `mech_e2.py` |
| **clock-prior** (rebuilt) | `prior/<model>/{conversion_test.csv,describe.csv}` (required by the shared setup cell; E2 itself does not read it) |

**Settings:** GPU T4 x2, Internet ON, Save & Run All. **Output:** `mech_F2_results.zip`. Runtime: a GUESS of about 3-6 h (descriptive; V3 informational for this model); read the first-pair extrapolation.


In [ ]:
# CELL 1 -- GPU CHECK: stop the notebook right here if CUDA is not available
import torch, subprocess

MIN_GPUS = 2
if not torch.cuda.is_available():
    raise RuntimeError("STOP: CUDA is not available. Settings -> Accelerator -> 'GPU T4 x2', then Save & Run All again.")
n_gpu = torch.cuda.device_count()
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}  {p.name}  {p.total_memory / 2**30:.1f} GiB")
if n_gpu < MIN_GPUS:
    raise RuntimeError(f"STOP: this notebook needs {MIN_GPUS} GPU(s) but only {n_gpu} visible. Choose 'GPU T4 x2'.")
subprocess.run(["nvidia-smi"])
print("GPU check passed.")


In [ ]:
# CELL 2 -- COPY INPUTS from the attached Kaggle datasets into a clean working folder
import os, shutil, sys, time, json, subprocess, zipfile, datetime, hashlib

T_START = time.time()
INPUT, WORK = "/kaggle/input", "/kaggle/working"
RUN = f"{WORK}/run"
OUT_ZIP = f"{WORK}/mech_F2_results.zip"
CODE_FILES = ["adapters.py", "clocks.py", "eval_behavior.py", "describe_check.py", "text_only_v2.py", "conversion_test.py", "cap_check.py", "mech_lib.py", "mech_e3.py", "mech_e1.py", "mech_e2.py", "textnorm.py"]
PRIOR_MODELS = ["qwen2.5-vl-3b", "qwen2.5-vl-7b", "internvl3-2b", "internvl3-8b"]

shutil.rmtree(RUN, ignore_errors=True)
os.makedirs(f"{RUN}/logs")

def find_root(marker, maxdepth=4):
    """Directory under /kaggle/input that contains `marker` (a relative path). Dataset slugs don't matter."""
    base = INPUT.rstrip("/").count("/")
    for dirpath, dirnames, _ in os.walk(INPUT):
        if dirpath.count("/") - base > maxdepth:
            dirnames[:] = []
            continue
        if os.path.exists(os.path.join(dirpath, marker)):
            return dirpath
    raise FileNotFoundError(f"Could not find '{marker}' under {INPUT}. Attach the dataset that contains it "
                            f"(see the checklist at the top of this notebook).")

code_root = find_root("adapters.py")
for f in CODE_FILES:
    shutil.copy(os.path.join(code_root, f), RUN)
print("code from:", code_root)

prior_root = find_root("prior/qwen2.5-vl-3b/describe.csv")
shutil.copytree(os.path.join(prior_root, "prior"), f"{RUN}/prior")
for m in PRIOR_MODELS:
    for f in ("conversion_test.csv", "describe.csv"):
        p = f"{RUN}/prior/{m}/{f}"
        assert os.path.exists(p), f"missing {p} -- the clock-prior dataset must contain prior/<model>/{f}"
    n_conv = len(__import__("pandas").read_csv(f"{RUN}/prior/{m}/conversion_test.csv"))
    print(f"prior/{m}: conversion_test rows={n_conv}")
print("prior results from:", prior_root)
print("inputs copied OK.")
du = shutil.disk_usage("/"); print(f"disk: {du.free / 2**30:.1f} GiB free of {du.total / 2**30:.1f} GiB")


In [ ]:
# CELL 3 -- ENVIRONMENT: env vars, versions, internet check (model weights are downloaded from Hugging Face)
os.environ.update({"PYTHONUNBUFFERED": "1", "TQDM_MININTERVAL": "60", "HF_HUB_DISABLE_PROGRESS_BARS": "1",
                   "TOKENIZERS_PARALLELISM": "false", "PYTHONIOENCODING": "utf-8", "PYTHONUTF8": "1"})
r = subprocess.run([sys.executable, "-c",
    "import torch, transformers, accelerate, torchvision, matplotlib, pandas, PIL, sys;"
    "print('python', sys.version.split()[0]);"
    "[print(m.__name__, m.__version__) for m in (torch, transformers, accelerate, torchvision, matplotlib, pandas, PIL)];"
    "print('cuda', torch.version.cuda, '| gpus', torch.cuda.device_count())"],
    capture_output=True, text=True)
print(r.stdout, r.stderr[-800:])
assert r.returncode == 0, "a required package is missing -- see the error above"
open(f"{RUN}/env.txt", "w").write(r.stdout + "\nstarted " + datetime.datetime.utcnow().isoformat() + "Z\n")
import urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=20)
    print("internet OK (huggingface.co reachable)")
except Exception as e:
    raise RuntimeError(f"STOP: huggingface.co not reachable ({e}). Settings -> Internet -> ON.")


In [ ]:
# CELL 4 -- THE STEPS: (name, command, guessed_minutes)
M3B, A3B = "Qwen/Qwen2.5-VL-3B-Instruct", "qwen2.5-vl-3b"
M2B, A2B = "OpenGVLab/InternVL3-2B-hf", "internvl3-2b"
M7B, A7B = "Qwen/Qwen2.5-VL-7B-Instruct", "qwen2.5-vl-7b"
M8B, A8B = "OpenGVLab/InternVL3-8B-hf", "internvl3-8b"
DEADLINE_H = 10.5      # do not START a new step after this many hours
HARD_END_H = 11.5      # do not start a step whose guessed duration would end after this

def e3(model_id, adapter, parts, extra=()):
    return ["python", "mech_e3.py", "--model_id", model_id, "--adapter", adapter, "--parts", parts, *extra]

def purge_cache(*repos):
    code = ("import shutil,os,glob;"
            "[shutil.rmtree(p, ignore_errors=True) for r in %r for p in glob.glob(os.path.expanduser('~/.cache/huggingface/hub/models--'+r))]" % (list(repos),))
    return ["python", "-c", code]

RPQ = ("--rep_penalty", "1.0")      # Qwen models only
def e2(model_id, adapter, parts, extra=()):
    return ["python", "mech_e2.py", "--model_id", model_id, "--adapter", adapter, "--parts", parts, *extra]
STEPS = [
    ("01_i8b_E2_standard",    e2(M8B, A8B, "verify,run", ("--variants", "standard", "--v3_informational")), 360),
    ("02_purge_8b_cache",     purge_cache("OpenGVLab--InternVL3-8B-hf"), 1),
]
for name, cmd, est in STEPS:
    print(f"{name:<32} ~{est:>3} min (guess)  {' '.join(cmd)[:110]}")
print(f"guessed total: {sum(e for _, _, e in STEPS) / 60:.1f} h")


In [ ]:
# CELL 5 -- RUN every step. A failing step is recorded and the notebook CONTINUES with the next one.
EXPECTED = ['results/mech_e2/internvl3-8b/verify.json', 'results/mech_e2/internvl3-8b/e2_trials.csv', 'results/mech_e2/internvl3-8b/e2_pairs.csv']        # files that should exist at the end (relative to the run folder)
status = []

def make_zip():
    """Whitelisted snapshot of everything worth keeping; refreshed after EVERY step."""
    tmp = OUT_ZIP + ".tmp"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for sub in ("results", "logs"):
            for dp, _, fns in os.walk(f"{RUN}/{sub}"):
                for fn in fns:
                    full = os.path.join(dp, fn); z.write(full, os.path.relpath(full, RUN))
        for rel in ["status.json", "STATUS.txt", "env.txt", "conversion_images/manifest.csv",
                    ] + CODE_FILES:
            if os.path.exists(f"{RUN}/{rel}"):
                z.write(f"{RUN}/{rel}", rel)
    os.replace(tmp, OUT_ZIP)

def run_step(name, cmd, est_min):
    elapsed_h = (time.time() - T_START) / 3600
    if elapsed_h > DEADLINE_H or elapsed_h + est_min / 60 > HARD_END_H:
        return {"step": name, "status": "SKIPPED (time budget)", "returncode": None, "minutes": 0.0}
    cmd = [sys.executable if c == "python" else c for c in cmd]
    print(f"\n{'=' * 100}\n[{datetime.datetime.utcnow():%H:%M:%S}] START {name}  (elapsed {elapsed_h:.2f} h)\n{'=' * 100}", flush=True)
    t0 = time.time()
    with open(f"{RUN}/logs/{name}.log", "w", encoding="utf-8") as log:
        p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             encoding="utf-8", errors="replace", bufsize=1)
        for line in p.stdout:
            print(line, end="", flush=True); log.write(line)
        rc = p.wait()
    mins = (time.time() - t0) / 60
    print(f"[{name}] finished rc={rc} in {mins:.1f} min", flush=True)
    return {"step": name, "status": "OK" if rc == 0 else f"FAILED rc={rc}", "returncode": rc, "minutes": round(mins, 1)}

for name, cmd, est in STEPS:
    try:
        status.append(run_step(name, cmd, est))
    except Exception as e:           # never let one step's crash stop the notebook
        status.append({"step": name, "status": f"ERROR {type(e).__name__}: {e}"[:120], "returncode": None, "minutes": 0.0})
        print(f"[{name}] notebook-side error: {e}", flush=True)
    json.dump(status, open(f"{RUN}/status.json", "w"), indent=1)
    make_zip()                       # insurance: the zip is valid after every step
    print(f"zip refreshed after {name}: {os.path.getsize(OUT_ZIP) / 1e6:.1f} MB", flush=True)


In [ ]:
# CELL 6 -- FINAL: status table, completeness check, ONE output zip
lines = [f"{s['step']:<32} {s['status']:<26} {s['minutes']:>7} min" for s in status]
missing = [f for f in EXPECTED if not os.path.exists(f"{RUN}/{f}")]
lines += ["", f"total wall time: {(time.time() - T_START) / 3600:.2f} h",
          "MISSING expected files: " + (", ".join(missing) if missing else "none")]
open(f"{RUN}/STATUS.txt", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))
make_zip()
with zipfile.ZipFile(OUT_ZIP) as z:
    assert z.testzip() is None, "zip is corrupt"
    names = z.namelist()
print(f"\nOUTPUT: {OUT_ZIP}  ({os.path.getsize(OUT_ZIP) / 1e6:.1f} MB, {len(names)} files)")
for n in sorted(names):
    if n.startswith("results/") and n.endswith((".csv", ".json", ".png")):
        print("  ", n)
shutil.rmtree(RUN)      # leave ONLY the zip in /kaggle/working (the zip is verified above)
print("\nDONE." if not missing and all(s["status"] == "OK" for s in status) else
      "\nDONE WITH PROBLEMS -- read STATUS.txt inside the zip and the logs/ folder.")
